# Deepfake Audio Detection
This notebook covers the full pipeline for training and evaluating a CNN model to classify speech recordings as Genuine or Deepfake.

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve

# Ensure memory growth for GPU if available
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

2026-06-14 18:18:13.904295: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781461093.928207     553 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781461093.936076     553 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781461093.955358     553 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781461093.955378     553 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781461093.955381     553 computation_placer.cc:177] computation placer alr

## 1. Data Pipeline and Feature Extraction

In [2]:
SAMPLING_RATE = 16000
MAX_DURATION = 3  # seconds
MAX_SAMPLES = SAMPLING_RATE * MAX_DURATION
BATCH_SIZE = 32

def load_and_preprocess_audio(file_path, label):
    audio_binary = tf.io.read_file(file_path)
    audio, sample_rate = tf.audio.decode_wav(audio_binary)
    audio = tf.squeeze(audio, axis=-1)
    
    audio_len = tf.shape(audio)[0]
    if audio_len < MAX_SAMPLES:
        padding = [[0, MAX_SAMPLES - audio_len]]
        audio = tf.pad(audio, padding, "CONSTANT")
    else:
        audio = audio[:MAX_SAMPLES]
        
    stft = tf.signal.stft(audio, frame_length=512, frame_step=256)
    spectrogram = tf.abs(stft)
    
    # CRITICAL FIX: Logarithmic compression
    spectrogram = tf.math.log(spectrogram + 1e-6)
    
    spectrogram = tf.expand_dims(spectrogram, -1)
    return spectrogram, label

def create_dataset(directory, batch_size=32):
    fake_dir = os.path.join(directory, 'fake')
    real_dir = os.path.join(directory, 'real')
    
    fake_files = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.endswith('.wav')]
    real_files = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.endswith('.wav')]
    
    file_paths = fake_files + real_files
    labels = [0] * len(fake_files) + [1] * len(real_files)
    
    indices = np.arange(len(file_paths))
    np.random.shuffle(indices)
    file_paths = np.array(file_paths)[indices]
    labels = np.array(labels)[indices]
    
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    ds = ds.map(load_and_preprocess_audio, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

## 2. Define Model Architecture

In [3]:
def build_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        layers.Conv2D(32, 3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        
        layers.Conv2D(64, 3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        
        layers.Conv2D(128, 3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        
        layers.Conv2D(256, 3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        
        # CRITICAL FIX: Replaced GlobalAveragePooling with Flatten
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), # Lowered learning rate
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

## 3. Train Model

In [4]:
train_dir = "/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm/training"
val_dir = "/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm/validation"

train_ds = create_dataset(train_dir, BATCH_SIZE)
val_ds = create_dataset(val_dir, BATCH_SIZE)

for spec, label in train_ds.take(1):
    input_shape = spec.shape[1:]

model = build_model(input_shape)
model.summary()

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath="/kaggle/working/best_model.keras", 
    monitor='val_accuracy', 
    save_best_only=True
)

# Note: Uncomment to train the model
history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stopping, model_checkpoint])

I0000 00:00:1781461098.347320     553 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1781461098.349309     553 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 186, 257, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 186, 257, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 186, 257, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 93, 128, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 93, 128, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 93, 128, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 93, 128, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 46, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 46, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 46, 64, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 46, 64, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 23, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 23, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 23, 32, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 23, 32, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 11, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 45056)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     5,767,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,157,185 (23.49 MB)

 Trainable params: 6,156,225 (23.48 MB)

...

   2/1684 ━━━━━━━━━━━━━━━━━━━━ 2:20 83ms/step - accuracy: 0.5234 - loss: 1.9456  

I0000 00:00:1781461112.354236     613 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1683/1684 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.9438 - loss: 0.1764

2026-06-14 18:20:31.953313: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-14 18:20:32.121092: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


1684/1684 ━━━━━━━━━━━━━━━━━━━━ 148s 81ms/step - accuracy: 0.9762 - loss: 0.0702 - val_accuracy: 0.9656 - val_loss: 0.1180
Epoch 2/10
1684/1684 ━━━━━━━━━━━━━━━━━━━━ 125s 74ms/step - accuracy: 0.9954 - loss: 0.0147 - val_accuracy: 0.9958 - val_loss: 0.0145
Epoch 3/10
1684/1684 ━━━━━━━━━━━━━━━━━━━━ 125s 74ms/step - accuracy: 0.9971 - loss: 0.0104 - val_accuracy: 0.9913 - val_loss: 0.0265
Epoch 4/10
1684/1684 ━━━━━━━━━━━━━━━━━━━━ 124s 74ms/step - accuracy: 0.9975 - loss: 0.0081 - val_accuracy: 0.9645 - val_loss: 0.1694
Epoch 5/10
1684/1684 ━━━━━━━━━━━━━━━━━━━━ 125s 74ms/step - accuracy: 0.9982 - loss: 0.0060 - val_accuracy: 0.9994 - val_loss: 0.0046
Epoch 6/10
1684/1684 ━━━━━━━━━━━━━━━━━━━━ 124s 73ms/step - accuracy: 0.9974 - loss: 0.0079 - val_accuracy: 0.9996 - val_loss: 0.0041
Epoch 7/10
1684/1684 ━━━━━━━━━━━━━━━━━━━━ 124s 74ms/step - accuracy: 0.9985 - loss: 0.0044 - val_accuracy: 0.9996 - val_loss: 0.0046
Epoch 8/10
1684/1684 ━━━━━━━━━━━━━━━━━━━━ 123s 73ms/step - accuracy: 0.9988 - lo

## 4. Evaluation

In [7]:
test_dir = "/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm/testing"
test_ds = create_dataset(test_dir, BATCH_SIZE)

# Extract true labels for evaluation
y_true = []
for _, labels in test_ds:
    y_true.extend(labels.numpy())
y_true = np.array(y_true)

model = tf.keras.models.load_model(r"/kaggle/working/best_model.keras")
y_pred_probs = model.predict(test_ds).flatten()

# CALIBRATION FIX: Automatically find the perfect threshold using ROC
fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)
fnr = 1 - tpr
eer_idx = np.nanargmin(np.absolute((fnr - fpr)))
eer = fpr[eer_idx]
optimal_threshold = thresholds[eer_idx]

# Make final decisions using the optimized threshold instead of a hardcoded 0.5
y_pred = (y_pred_probs >= optimal_threshold).astype(int)

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print(f"Overall Accuracy: {acc*100:.2f}%")
print(f"EER: {eer*100:.2f}%")
print(f"Optimal Decision Threshold: {optimal_threshold:.4f}")
print(f"F1 Score: {f1*100:.2f}%")
print("Confusion Matrix:")
print(cm)

145/145 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step
Overall Accuracy: 94.30%
EER: 5.70%
Optimal Decision Threshold: 1.0000
F1 Score: 94.18%
Confusion Matrix:
[[2235  135]
 [ 129 2135]]
